In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, KFold

from spectral_core import DataSplitter, build_candidates, compute_metrics
from spectral_core.models import SpectralData

/home/mykola/.cache/pypoetry/virtualenvs/spectral-core-S9wPqFE3-py3.11/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
DATA_DIR = Path('../data/processed/')
MODEL_DIR = Path('../data/models/')
RESULTS_PATH = MODEL_DIR / 'model_comparison.json'

RANDOM_STATE = 34
REPEAT_SEEDS = tuple(range(5))
BOOTSTRAP_DRAWS = 10_000

PAPER_METRICS = {
    'PLSR': {'r2': 0.76, 'rmse': 1.15, 'mae': 0.89},
    'CNN': {'r2': 0.73, 'rmse': 1.22, 'mae': 0.94},
    'Curve fitting': {'r2': 0.59, 'rmse': 1.51, 'mae': 1.11}
}

MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def load_array(name):
    return np.load(DATA_DIR / f'{name}.npy')


X_cal = np.vstack([load_array('X_train'), load_array('X_val')])
y_cal = np.concatenate([load_array('y_train'), load_array('y_val')])
X_test = load_array('X_test')
y_test = load_array('y_test')
wavenumbers = load_array('wavenumbers')

train_idx = load_array('train_idx')
val_idx = load_array('val_idx')
test_idx = load_array('test_idx')

n_all = len(train_idx) + len(val_idx) + len(test_idx)
X_all = np.empty((n_all, X_cal.shape[1]), dtype=X_cal.dtype)
y_all = np.empty(n_all, dtype=y_cal.dtype)
X_all[np.concatenate([train_idx, val_idx])] = X_cal
y_all[np.concatenate([train_idx, val_idx])] = y_cal
X_all[test_idx] = X_test
y_all[test_idx] = y_test

X_cal.shape, X_test.shape, X_all.shape

((538, 3319), (135, 3319), (673, 3319))

In [4]:
def fit_all(label, X_fit, y_fit, X_eval, y_eval):
    rows = {}
    predictions = {}
    estimators = {}

    for name, (estimator, grid, folds) in build_candidates(X_fit.shape[1]).items():
        search = GridSearchCV(
            estimator,
            grid,
            cv=KFold(n_splits=folds, shuffle=True, random_state=RANDOM_STATE),
            scoring='neg_root_mean_squared_error',
            n_jobs=-1
        )
        search.fit(X_fit, y_fit)
        y_pred = np.asarray(search.predict(X_eval)).ravel()

        predictions[name] = y_pred
        estimators[name] = search.best_estimator_
        rows[name] = {
            'representation': label,
            'model': name,
            'rmsecv': float(-search.best_score_),
            **compute_metrics(y_eval, y_pred)
        }
        print(f"  {name:<16} R2 {rows[name]['r2']:.3f}  RMSE {rows[name]['rmse']:.3f}")

    return rows, predictions, estimators

In [5]:
print(f'full spectra: {X_cal.shape[1]} features')
full_rows, full_predictions, full_estimators = fit_all(
    'full spectra', X_cal, y_cal, X_test, y_test
)

full spectra: 3319 features


/home/mykola/.cache/pypoetry/virtualenvs/spectral-core-S9wPqFE3-py3.11/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
/home/mykola/.cache/pypoetry/virtualenvs/spectral-core-S9wPqFE3-py3.11/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
/home/mykola/.cache/pypoetry/virtualenvs/spectral-core-S9wPqFE3-py3.11/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
/home/mykola/.cache/pypoetry/virtualenvs/spectral-core-S9wPqFE3-py3.11/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scal

  PLSR             R2 0.755  RMSE 1.165
  SVR (RBF)        R2 0.716  RMSE 1.252
  Random Forest    R2 0.692  RMSE 1.305
  XGBoost          R2 0.695  RMSE 1.298


In [6]:
results = pd.DataFrame(list(full_rows.values()))
results[['model', 'r2', 'r', 'mae', 'mse', 'rmse', 'rmsecv']].sort_values(
    'r2', ascending=False
).reset_index(drop=True)

,model,r2,r,mae,mse,rmse,rmsecv
0,PLSR,0.754616,0.870453,0.885128,1.356571,1.164719,1.280014
1,SVR (RBF),0.716261,0.852254,0.956026,1.568611,1.252442,1.232303
2,XGBoost,0.695246,0.835136,0.954237,1.684790,1.297995,1.303970
3,Random Forest,0.691957,0.836011,0.961012,1.702974,1.304980,1.382051


In [7]:
rng = np.random.default_rng(RANDOM_STATE)
draws = rng.integers(0, len(y_test), size=(BOOTSTRAP_DRAWS, len(y_test)))


def bootstrap_rmse(y_pred):
    squared = (y_test - y_pred) ** 2
    return np.sqrt(squared[draws].mean(axis=1))


reference = bootstrap_rmse(full_predictions['PLSR'])
comparison = []
for name, y_pred in full_predictions.items():
    if name == 'PLSR':
        continue
    difference = bootstrap_rmse(y_pred) - reference
    comparison.append({
        'model': name,
        'rmse_minus_plsr': float(difference.mean()),
        'ci_low': float(np.percentile(difference, 2.5)),
        'ci_high': float(np.percentile(difference, 97.5)),
        'p_worse_than_plsr': float((difference > 0).mean())
    })

pd.DataFrame(comparison)

,model,rmse_minus_plsr,ci_low,ci_high,p_worse_than_plsr
0,SVR (RBF),0.088024,-0.001926,0.175590,0.9715
1,Random Forest,0.138310,-0.025128,0.325870,0.9476
2,XGBoost,0.132995,-0.004850,0.282079,0.9696


In [8]:
repeat_rows = []
for seed in REPEAT_SEEDS:
    splitter = DataSplitter(test_size=0.2, stratify_bins=8, random_state=seed)
    split = splitter.train_test_split(
        SpectralData(spectra=X_all, wavenumbers=wavenumbers, features={'HbA1c': y_all})
    )
    X_fit, y_fit = split.train.spectra, split.train.get_feature('HbA1c')
    X_eval, y_eval = split.test.spectra, split.test.get_feature('HbA1c')

    for name, estimator in full_estimators.items():
        model = clone(estimator).fit(X_fit, y_fit)
        repeat_rows.append({
            'seed': seed,
            'model': name,
            **compute_metrics(y_eval, model.predict(X_eval))
        })
    print(f'seed {seed} done')

repeats = pd.DataFrame(repeat_rows)
repeats.groupby('model')['r2'].agg(['mean', 'std', 'min', 'max']).sort_values('mean', ascending=False)

seed 0 done
seed 1 done
seed 2 done
seed 3 done
seed 4 done


,mean,std,min,max
model,,,,
SVR (RBF),0.693921,0.031846,0.665672,0.746886
PLSR,0.683689,0.043909,0.632774,0.753119
XGBoost,0.663178,0.063480,0.570529,0.729054
Random Forest,0.642837,0.072401,0.546462,0.718431


In [10]:
RESULTS_PATH.write_text(json.dumps({
    'single_split': results.to_dict(orient='records'),
    'bootstrap_vs_plsr': comparison,
    'repeated_splits': repeat_rows,
    'paper': PAPER_METRICS
}, indent=2))

for name, estimator in full_estimators.items():
    slug = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    joblib.dump(estimator, MODEL_DIR / f'{slug}_full_spectra.joblib')

print(f'results  {RESULTS_PATH}')
print(f'models   {MODEL_DIR}')

results  ../data/models/model_comparison.json
models   ../data/models
